# ZelusBench: Sustained Attention

**Does accuracy degrade as dependency chains grow longer?**

This task measures sustained attention by varying the number of dependency hops
between the origin and the queried point. The math at each step is trivial
(vector addition, rotation). What makes it hard is tracking a growing chain of
dependencies across the prompt.

Difficulty levels:
- **Shallow** (depth 2-3): A -> B -> C
- **Medium** (depth 5-6): longer transitive chains
- **Deep** (depth 9-10): long chains that stress working memory

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import re
import os

os.environ["RENDER_SUBRUNS"] = "False"

In [ ]:
# ZelusBench engine
from zelusbench.scenarios.config import (
    ScenarioConfig, DAGStructure, QueryType,
    DistractorLevel, TransformDensity,
)
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.parser import parse_model_response
from zelusbench.evaluation.scorer import score_query

In [ ]:
def score_response(response_text, scenario):
    """Parse model response and score each query against ground truth."""
    query_dicts = [q.to_dict() for q in scenario.queries]
    parts = re.split(r'\[Query\s+q_\d+\]', response_text)
    if len(parts) > 1:
        parts = parts[1:]
    if len(parts) != len(query_dicts):
        parts = [response_text] * len(query_dicts)

    scores = []
    for qd, rp in zip(query_dicts, parts):
        parsed = parse_model_response(rp, qd)
        scores.append(score_query(qd, parsed))
    return scores

## Sub-task: Single Scenario

In [ ]:
@kbench.task(store_task=False)
def sustained_scenario(llm, chain_depth: int, seed: int) -> float:
    """Run one sustained-attention scenario at a given chain depth."""
    cfg = ScenarioConfig(
        dim=3,
        min_chain_depth=chain_depth, max_chain_depth=chain_depth,
        dag_structure=DAGStructure.LINEAR,
        distractor_level=DistractorLevel.CLEAN,
        transform_density=TransformDensity.STATIC,
        query_types=[QueryType.POSITION, QueryType.DISTANCE],
        num_queries=3, num_splits=3,
        seed=seed,
    )
    gen = ScenarioGenerator(cfg)
    scenario = gen.generate(f"sustained_d{chain_depth}_s{seed}")

    response = llm.prompt(scenario.prompt)
    scores = score_response(response, scenario)

    avg = float(np.mean([s.score for s in scores]))

    for s in scores:
        kbench.assertions.assert_true(
            s.score > 0,
            expectation=(
                f"{s.query_id} (depth={s.chain_depth}): "
                f"model should track dependency chain. "
                f"Tier={s.tier.name}, error={s.relative_error}"
            )
        )

    return avg

## Main Task: Sustained Attention Evaluation

In [ ]:
eval_data = pd.DataFrame([
    {"chain_depth": depth, "seed": seed}
    for depth in [2, 3, 5, 6, 9, 10]
    for seed in range(5)
])

print(f"Evaluation matrix: {len(eval_data)} scenarios")
print(f"Chain depths: {sorted(eval_data.chain_depth.unique())}")
print(f"Seeds per depth: {eval_data.groupby('chain_depth').size().iloc[0]}")

In [ ]:
@kbench.task(name="zelusbench_sustained_attention")
def zelusbench_sustained_attention(llm) -> tuple[float, float]:
    """Sustained attention: accuracy vs chain depth.

    Returns (mean_accuracy, std_dev) across all depth levels.
    """
    with kbench.client.enable_cache():
        runs = sustained_scenario.evaluate(
            llm=[llm],
            evaluation_data=eval_data,
            n_jobs=2,
            timeout=180,
            remove_run_files=True,
            stop_condition=lambda r: len(r) == len(eval_data),
            max_attempts=60,
            retry_delay=10,
        )

    results_df = runs.as_dataframe()
    scores = results_df["result"].dropna().tolist()

    accuracy = float(np.mean(scores)) if scores else 0.0
    std = float(np.std(scores)) if scores else 0.0

    kbench.assertions.assert_true(
        len(scores) > 0,
        expectation="At least some scenarios should produce results"
    )

    return accuracy, std

In [ ]:
run = zelusbench_sustained_attention.run(kbench.llm)
run

In [ ]:
%choose zelusbench_sustained_attention